# Automated Regional Impact Auditor (ARIA)
## River Flood Shelter Risk Assessment

**Captain's Log**: Starting analysis of Taiwan's flood risk for emergency shelters using WRA river data and Fire Agency shelter locations.

In [ ]:
# Import required libraries
import geopandas as gpd
import pandas as pd
import folium
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv
from urllib.parse import quote
import json

# Load environment variables
load_dotenv()

# Buffer distances from .env
BUFFER_HIGH = int(os.getenv('BUFFER_HIGH', 500))
BUFFER_MED = int(os.getenv('BUFFER_MED', 1000))
BUFFER_LOW = int(os.getenv('BUFFER_LOW', 2000))

print(f"Buffer distances: High={BUFFER_HIGH}m, Medium={BUFFER_MED}m, Low={BUFFER_LOW}m")

## 1. Data Loading & Cleaning

**Captain's Log**: Loading river polygons from WRA API and checking coordinate system.

In [ ]:
# Load river data from WRA
rivers_url = 'https://gic.wra.gov.tw/Gis/gic/API/Google/DownLoad.aspx?fname=RIVERPOLY&filetype=SHP'
rivers = gpd.read_file(rivers_url)

print(f"River data loaded: {len(rivers)} polygons")
print(f"Original CRS: {rivers.crs}")
print(f"Bounds: {rivers.total_bounds}")

# Ensure we're working in EPSG:3826 (Taiwan datum)
if rivers.crs != 'EPSG:3826':
    rivers = rivers.to_crs('EPSG:3826')
    print(f"Converted to CRS: {rivers.crs}")

**Captain's Log**: Loading shelter data from Fire Agency CSV and performing coordinate validation.

In [ ]:
# Load shelter data (you'll need to download this from data.gov.tw)
# Download from: https://data.gov.tw/dataset/73242
# Save as: data/避難收容處所.csv

try:
    shelters_csv = pd.read_csv('data/避難收容處所.csv', encoding='utf-8')
except UnicodeDecodeError:
    shelters_csv = pd.read_csv('data/避難收容處所.csv', encoding='big5')

print(f"Original shelter records: {len(shelters_csv)}")
print(f"Columns: {list(shelters_csv.columns)}")

# Display first few rows to understand the data structure
shelters_csv.head()

In [ ]:
# Data cleaning - remove invalid coordinates
# Assuming coordinate columns are named '經度' and '緯度' (may need adjustment)
lon_col = '經度'  # Adjust based on actual column names
lat_col = '緯度'  # Adjust based on actual column names

# Check if these columns exist
if lon_col not in shelters_csv.columns or lat_col not in shelters_csv.columns:
    print("Available columns:")
    for col in shelters_csv.columns:
        print(f"  - {col}")
    # You may need to adjust column names based on the actual data
else:
    # Filter valid coordinates (Taiwan bounds: lon 119-122, lat 21-26)
    valid_shelters = shelters_csv[
        (shelters_csv[lon_col] >= 119) & (shelters_csv[lon_col] <= 122) &
        (shelters_csv[lat_col] >= 21) & (shelters_csv[lat_col] <= 26) &
        (shelters_csv[lon_col] != 0) & (shelters_csv[lat_col] != 0)
    ].copy()
    
    print(f"Valid shelter records after cleaning: {len(valid_shelters)}")
    print(f"Removed {len(shelters_csv) - len(valid_shelters)} invalid records")
    
    # Convert to GeoDataFrame
    shelters = gpd.GeoDataFrame(
        valid_shelters,
        geometry=gpd.points_from_xy(valid_shelters[lon_col], valid_shelters[lat_col]),
        crs='EPSG:4326'
    )
    
    # Convert to EPSG:3826 for buffer analysis
    shelters = shelters.to_crs('EPSG:3826')
    print(f"Shelters CRS: {shelters.crs}")
    print(f"Shelter bounds: {shelters.total_bounds}")

**Captain's Log**: Loading township boundaries for regional analysis.

In [ ]:
# Load township boundaries
township_url = 'https://www.tgos.tw/tgos/VirtualDir/Product/3fe61d4a-ca23-4f45-8aca-4a536f40f290/' + quote('鄉(鎮、市、區)界線1140318.zip')
townships = gpd.read_file(township_url)
townships = townships.to_crs('EPSG:3826')

print(f"Township boundaries loaded: {len(townships)} polygons")
print(f"Township CRS: {townships.crs}")
print(f"Available columns: {list(townships.columns)}")

## 2. Multi-Level Buffer Analysis

**Captain's Log**: Creating three-tier buffer zones around rivers for risk assessment.

In [ ]:
# Create multi-level buffer zones
river_buffers_high = rivers.buffer(BUFFER_HIGH)
river_buffers_med = rivers.buffer(BUFFER_MED)
river_buffers_low = rivers.buffer(BUFFER_LOW)

# Convert to GeoDataFrame
buffer_high_gdf = gpd.GeoDataFrame(geometry=river_buffers_high, crs='EPSG:3826')
buffer_med_gdf = gpd.GeoDataFrame(geometry=river_buffers_med, crs='EPSG:3826')
buffer_low_gdf = gpd.GeoDataFrame(geometry=river_buffers_low, crs='EPSG:3826')

# Dissolve to create unified buffer zones
buffer_high_unified = buffer_high_gdf.dissolve()
buffer_med_unified = buffer_med_gdf.dissolve()
buffer_low_unified = buffer_low_gdf.dissolve()

print(f"High risk buffer (500m): {len(buffer_high_unified)} zones")
print(f"Medium risk buffer (1km): {len(buffer_med_unified)} zones")
print(f"Low risk buffer (2km): {len(buffer_low_unified)} zones")

**Captain's Log**: Performing spatial joins to identify shelters in each risk zone.

In [ ]:
# Spatial joins to identify shelters in each buffer zone
shelters_in_high = gpd.sjoin(shelters, buffer_high_unified, how='inner', predicate='intersects')
shelters_in_med = gpd.sjoin(shelters, buffer_med_unified, how='inner', predicate='intersects')
shelters_in_low = gpd.sjoin(shelters, buffer_low_unified, how='inner', predicate='intersects')

print(f"Shelters in high risk zone (500m): {len(shelters_in_high)}")
print(f"Shelters in medium risk zone (1km): {len(shelters_in_med)}")
print(f"Shelters in low risk zone (2km): {len(shelters_in_low)}")

# Assign risk levels (highest priority wins)
shelters['risk_level'] = 'safe'  # Default to safe

# High risk overrides all
shelters.loc[shelters.index.isin(shelters_in_high.index), 'risk_level'] = 'high'
# Medium risk overrides low and safe
shelters.loc[shelters.index.isin(shelters_in_med.index), 'risk_level'] = 'medium'
# Low risk overrides safe
shelters.loc[shelters.index.isin(shelters_in_low.index), 'risk_level'] = 'low'

# Count shelters by risk level
risk_counts = shelters['risk_level'].value_counts()
print("\nShelters by risk level:")
for level, count in risk_counts.items():
    print(f"  {level}: {count}")

## 3. Capacity Gap Analysis

**Captain's Log**: Analyzing shelter capacity gaps by administrative region.

In [ ]:
# Spatial join shelters with townships
shelters_with_township = gpd.sjoin(shelters, townships, how='inner', predicate='intersects')

# Group by township and risk level
township_analysis = shelters_with_township.groupby(['TOWNNAME', 'risk_level']).agg({
    'geometry': 'count',  # Count of shelters
    # Add capacity column name here (adjust based on actual data)
}).rename(columns={'geometry': 'shelter_count'}).reset_index()

print("Township risk analysis (sample):")
print(township_analysis.head(10))

In [ ]:
# Find top 10 most at-risk townships
high_risk_by_township = township_analysis[township_analysis['risk_level'] == 'high'].sort_values('shelter_count', ascending=False)

print("Top 10 Townships with Most High-Risk Shelters:")
print(high_risk_by_township.head(10))

## 4. Visualization

**Captain's Log**: Creating interactive risk map and statistical charts.

In [ ]:
# Create interactive map
# Calculate center of the study area
bounds = shelters.total_bounds
center_lat = (bounds[1] + bounds[3]) / 2
center_lon = (bounds[0] + bounds[2]) / 2

# Convert back to WGS84 for folium
shelters_wgs84 = shelters.to_crs('EPSG:4326')
rivers_wgs84 = rivers.to_crs('EPSG:4326')
buffer_high_wgs84 = buffer_high_unified.to_crs('EPSG:4326')
buffer_med_wgs84 = buffer_med_unified.to_crs('EPSG:4326')
buffer_low_wgs84 = buffer_low_unified.to_crs('EPSG:4326')
townships_wgs84 = townships.to_crs('EPSG:4326')

# Create base map
m = folium.Map(location=[center_lat, center_lon], zoom_start=8)

# Add buffer zones
folium.GeoJson(
    buffer_low_wgs84,
    style_function=lambda x: {'fillColor': 'yellow', 'color': 'none', 'fillOpacity': 0.2},
    name='Low Risk (2km)'
).add_to(m)

folium.GeoJson(
    buffer_med_wgs84,
    style_function=lambda x: {'fillColor': 'orange', 'color': 'none', 'fillOpacity': 0.3},
    name='Medium Risk (1km)'
).add_to(m)

folium.GeoJson(
    buffer_high_wgs84,
    style_function=lambda x: {'fillColor': 'red', 'color': 'none', 'fillOpacity': 0.4},
    name='High Risk (500m)'
).add_to(m)

# Add rivers
folium.GeoJson(
    rivers_wgs84,
    style_function=lambda x: {'color': 'blue', 'weight': 2},
    name='Rivers'
).add_to(m)

# Add shelters with risk-based colors
risk_colors = {
    'high': 'red',
    'medium': 'orange', 
    'low': 'yellow',
    'safe': 'green'
}

for idx, shelter in shelters_wgs84.iterrows():
    risk = shelter['risk_level']
    folium.CircleMarker(
        location=[shelter.geometry.y, shelter.geometry.x],
        radius=5,
        popup=f"Risk: {risk}\nName: {shelter.get('name', 'N/A')}",
        color=risk_colors.get(risk, 'gray'),
        fill=True,
        fillColor=risk_colors.get(risk, 'gray')
    ).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

# Save map
m.save('risk_map.html')
print("Interactive risk map saved as 'risk_map.html'")
m

In [ ]:
# Create bar chart of top 10 high-risk townships
top_10 = high_risk_by_township.head(10)

plt.figure(figsize=(12, 6))
plt.bar(top_10['TOWNNAME'], top_10['shelter_count'], color='red', alpha=0.7)
plt.title('Top 10 Townships with Most High-Risk Shelters')
plt.xlabel('Township')
plt.ylabel('Number of High-Risk Shelters')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('risk_chart.png', dpi=300, bbox_inches='tight')
plt.show()
print("Risk chart saved as 'risk_chart.png'")

## 5. Export Results

**Captain's Log**: Exporting shelter risk audit results.

In [ ]:
# Export shelter risk audit
shelter_audit = shelters[['risk_level']].copy()
# Add other relevant columns as needed
shelter_audit['shelter_id'] = shelters.index

# Save to JSON
shelter_audit.to_json('shelter_risk_audit.json', orient='records', indent=2)
print("Shelter risk audit exported to 'shelter_risk_audit.json'")

# Summary statistics
print("\n=== FINAL SUMMARY ===")
print(f"Total shelters analyzed: {len(shelters)}")
print(f"High risk shelters: {len(shelters[shelters['risk_level'] == 'high'])}")
print(f"Medium risk shelters: {len(shelters[shelters['risk_level'] == 'medium'])}")
print(f"Low risk shelters: {len(shelters[shelters['risk_level'] == 'low'])}")
print(f"Safe shelters: {len(shelters[shelters['risk_level'] == 'safe'])}")